# Microsoft Foundry Local 
## Practical Applications

<img src="https://github.com/retkowsky/foundry-local/blob/main/foundrylocal.jpg?raw=true">

This notebook demonstrates **end‑to‑end practical AI applications running fully locally** using **Foundry Local** and an **OpenAI‑compatible API**.  
It focuses on **real-world use cases** such as text summarization, batch classification, structured data extraction, and a simple Retrieval‑Augmented Generation (RAG) pipeline — all executed **on-device with no data leaving the machine**.

The notebook serves as a **hands-on reference and demo script** for enterprise, edge, and disconnected environments.

---

## Environment & Runtime Setup

- Python 3.12 (Anaconda distribution)
- Common data and utility libraries: `pandas`, `matplotlib`, `requests`, `json`, `statistics`
- Foundry Local runtime managed via `FoundryLocalManager`
- Local OpenAI‑compatible REST endpoint (no API key required)
- Local model cache stored on disk for offline reuse

The notebook starts and validates the Foundry Local service, exposing:
- Service URI
- OpenAI‑compatible `/v1` endpoint
- Local model cache location

---

## Model Catalog & Management

- Lists all available local model variants (80+)
- Converts model metadata into a clean `pandas` DataFrame
- Model attributes include:
  - Alias and model ID
  - Device type (CPU / GPU)
  - Execution provider (CUDA, OpenVINO, WebGPU, CPU)
  - Model size, license, supported tasks
  - Tool‑calling capability
- Demonstrates filtering models by alias (Qwen 2.5 family)
- Downloads and loads a selected model into memory
  - Example: `qwen2.5-0.5b-instruct-generic-cpu`

---

## Local OpenAI Client Initialization

- Initializes the `OpenAI` client using the local Foundry endpoint
- Fully compatible with standard OpenAI Chat Completions APIs
- Enables seamless portability from cloud to local execution

---

## Text Summarization Pipeline

Demonstrates a reusable **local text summarization function** with:
- Configurable summarization styles:
  - Concise
  - Bullet points
  - Simple / ELI5
- Adjustable output length
- Latency measurement per request

Use case highlights:
- Articles
- Reports
- Long-form documents
- Zero data egress and offline inference

---

## Batch Processing Pipeline

Implements efficient **batch inference** with:
- Progress tracking
- Per-item latency measurement
- Error handling and status reporting

Example use case:
- Sentiment classification of product reviews
- Output format: label + confidence score

Illustrates how local LLMs can scale across datasets while maintaining predictable performance.

---

## Structured Data Extraction (JSON)

Shows how to extract **strictly structured JSON** from unstructured text:
- Enforced schema via system prompts
- JSON‑only responses (no markdown, no explanations)
- Automatic parsing and validation

Example extraction tasks:
- Person profiles
- Product specifications
- Event details

Ideal for:
- ETL pipelines
- Document processing
- Automation workflows

---

## Simple Local RAG (Retrieval‑Augmented Generation)

Implements a **minimal local RAG system** using:
- Keyword‑based retrieval
- Local documents as a knowledge base
- Context‑grounded LLM responses only

Key characteristics:
- No vector database required
- Fully offline operation
- Clear separation between retrieval and generation
- Easy to replace retrieval with embeddings for production

Example queries:
- Hardware support
- Model capabilities
- Architecture behavior
- Offline execution

---

## Key Takeaways

- Demonstrates **practical, production‑style AI workflows** running entirely locally
- Uses **OpenAI‑compatible APIs** for easy migration from cloud to edge
- Covers multiple enterprise‑ready patterns:
  - Summarization
  - Classification
  - Structured extraction
  - RAG
- Ideal for:
  - Edge AI deployments
  - Disconnected or air‑gapped environments
  - Privacy‑sensitive workloads
  - Customer demos and technical workshops

This notebook can be used directly as a **customer demo**, **training asset**, or **reference implementation** for local AI application development with Foundry Local.

> **Docs:** [Foundry Local Documentation](https://learn.microsoft.com/azure/ai-foundry/foundry-local/)  
> **SDK Reference:** [Python SDK Reference](https://learn.microsoft.com/azure/ai-foundry/foundry-local/reference/reference-sdk?pivots=programming-language-python)

## Author

| Field | Details |
| --- | --- |
| Name | Serge Retkowsky |
| Email | serge.retkowsky@microsoft.com |
| LinkedIn | https://www.linkedin.com/in/serger/ |
| Medium publications | https://medium.com/@sergems18/ |

## Setup

In [2]:
import datetime
import GPUtil
import json
import matplotlib.pyplot as plt
import os
import pandas as pd
import platform
import psutil
import requests
import statistics
import sys
import time

from foundry_local import FoundryLocalManager
from openai import OpenAI

In [3]:
print(f"Python version: {sys.version}")

Python version: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]


In [3]:
print(f"Today is {datetime.datetime.today().strftime('%d-%b-%Y %H:%M:%S')}")

Today is 25-Feb-2026 13:01:09


In [4]:
print(f"💻 OS: {platform.system()} {platform.release()}")
print(f"- CPU: {platform.processor()}")
print(f"- CPU cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count()} logical")

ram = psutil.virtual_memory()
print(f"- RAM total:     {ram.total / (1024**3):.1f} GB")
print(f"- RAM available: {ram.available / (1024**3):.1f} GB")
print(f"- RAM used:      {ram.percent}%")

for part in psutil.disk_partitions():
    try:
        usage = psutil.disk_usage(part.mountpoint)
        print(f"\n💾 Disk [{part.device}] mounted on {part.mountpoint}")
        print(f"- Total: {usage.total / (1024**3):.1f} GB")
        print(f"- Used:  {usage.used / (1024**3):.1f} GB ({usage.percent}%)")
        print(f"- Free:  {usage.free / (1024**3):.1f} GB")
    except PermissionError:
        pass

gpus = GPUtil.getGPUs()

if not gpus:
    print("No GPU detected.")
else:
    for i, gpu in enumerate(gpus):
        print(f"\n🎮 GPU {i} — {gpu.name}")
        print(f"- VRAM Total : {gpu.memoryTotal:,.0f} MB")
        print(f"- VRAM Used  : {gpu.memoryUsed:,.0f} MB ({gpu.memoryUtil * 100:.0f}%)")
        print(f"- VRAM Free  : {gpu.memoryFree:,.0f} MB")
        print(f"- GPU Load   : {gpu.load * 100:.0f}%")
        print(f"- Temperature: {gpu.temperature} °C")

💻 OS: Windows 11
- CPU: Intel64 Family 6 Model 141 Stepping 1, GenuineIntel
- CPU cores: 8 physical, 16 logical
- RAM total:     63.7 GB
- RAM available: 31.5 GB
- RAM used:      50.5%

💾 Disk [C:\] mounted on C:\
- Total: 951.6 GB
- Used:  875.2 GB (92.0%)
- Free:  76.5 GB

🎮 GPU 0 — NVIDIA T1200 Laptop GPU
- VRAM Total : 4,096 MB
- VRAM Used  : 3,658 MB (89%)
- VRAM Free  : 278 MB
- GPU Load   : 33%
- Temperature: 62.0 °C


## Helper

In [5]:
def models_to_df(models):
    """Convert a list of FoundryModelInfo objects into a clean DataFrame."""
    return pd.DataFrame([{
        "alias": m.alias,
        "id": m.id,
        "device": m.device_type.value,
        "provider": m.execution_provider,
        "size_mb": m.file_size_mb,
        "tools": m.supports_tool_calling,
        "license": m.license,
        "task": m.task,
    } for m in models])

## Models

In [6]:
manager = FoundryLocalManager()
manager.start_service()

print(f"Service running : {manager.is_service_running()}")
print(f"Service URI     : {manager.service_uri}")
print(f"Endpoint (v1)   : {manager.endpoint}")
print(f"Cache location  : {manager.get_cache_location()}")

Service running : True
Service URI     : http://127.0.0.1:55311
Endpoint (v1)   : http://127.0.0.1:55311/v1
Cache location  : C:\models


In [7]:
os.listdir(manager.get_cache_location())

['Microsoft']

In [8]:
os.listdir(os.path.join(manager.get_cache_location(), "Microsoft"))

['Phi-4-mini-instruct-cuda-gpu-5']

In [9]:
catalog = manager.list_catalog_models()
print(f"Total model variants in catalog = {len(catalog)}")

df_catalog = models_to_df(catalog)
df_catalog

Total model variants in catalog = 80


,alias,id,device,provider,size_mb,tools,license,task
0,phi-4,Phi-4-cuda-gpu:1,GPU,CUDAExecutionProvider,8570,False,MIT,chat-completion
1,phi-4,phi-4-openvino-gpu:1,GPU,OpenVINOExecutionProvider,9046,False,MIT,chat-completion
2,phi-4,Phi-4-generic-gpu:1,GPU,WebGpuExecutionProvider,8570,False,MIT,chat-completion
3,phi-4,Phi-4-generic-cpu:1,CPU,CPUExecutionProvider,10403,False,MIT,chat-completion
4,phi-3.5-mini,Phi-3.5-mini-instruct-cuda-gpu:1,GPU,CUDAExecutionProvider,2181,False,MIT,chat-completion
...,...,...,...,...,...,...,...,...
75,qwen2.5-7b,qwen2.5-7b-instruct-generic-cpu:4,CPU,CPUExecutionProvider,6307,True,apache-2.0,chat-completion
76,whisper-large-v3-turbo,openai-whisper-large-v3-turbo-cuda-gpu:2,GPU,CUDAExecutionProvider,9000,False,apache-2.0,automatic-speech-recognition
77,whisper-large-v3-turbo,openai-whisper-large-v3-turbo-generic-cpu:2,CPU,CPUExecutionProvider,9000,False,apache-2.0,automatic-speech-recognition
78,gpt-oss-20b,gpt-oss-20b-generic-cpu:1,CPU,CPUExecutionProvider,12552,False,MIT,chat-completion


In [10]:
# Filter by alias keyword
df_catalog[df_catalog["alias"].str.contains("qwen", case=False, na=False)]

,alias,id,device,provider,size_mb,tools,license,task
36,qwen2.5-coder-0.5b,qwen2.5-coder-0.5b-instruct-cuda-gpu:4,GPU,CUDAExecutionProvider,528,True,apache-2.0,chat-completion
37,qwen2.5-coder-0.5b,qwen2.5-coder-0.5b-instruct-openvino-gpu:2,GPU,OpenVINOExecutionProvider,365,True,apache-2.0,chat-completion
38,qwen2.5-coder-0.5b,qwen2.5-coder-0.5b-instruct-generic-gpu:4,GPU,WebGpuExecutionProvider,528,True,apache-2.0,chat-completion
39,qwen2.5-coder-0.5b,qwen2.5-coder-0.5b-instruct-generic-cpu:4,CPU,CPUExecutionProvider,822,True,apache-2.0,chat-completion
44,qwen2.5-0.5b,qwen2.5-0.5b-instruct-cuda-gpu:4,GPU,CUDAExecutionProvider,528,True,apache-2.0,chat-completion
45,qwen2.5-0.5b,qwen2.5-0.5b-instruct-openvino-gpu:2,GPU,OpenVINOExecutionProvider,366,True,apache-2.0,chat-completion
46,qwen2.5-0.5b,qwen2.5-0.5b-instruct-generic-gpu:4,GPU,WebGpuExecutionProvider,700,True,apache-2.0,chat-completion
47,qwen2.5-0.5b,qwen2.5-0.5b-instruct-generic-cpu:4,CPU,CPUExecutionProvider,822,True,apache-2.0,chat-completion
48,qwen2.5-1.5b,qwen2.5-1.5b-instruct-cuda-gpu:4,GPU,CUDAExecutionProvider,1280,True,apache-2.0,chat-completion
49,qwen2.5-1.5b,qwen2.5-1.5b-instruct-openvino-gpu:2,GPU,OpenVINOExecutionProvider,1019,True,apache-2.0,chat-completion


In [11]:
# Filter by alias keyword
df_catalog[df_catalog["alias"].str.contains(
    "qwen2.5-0.5b", case=False, na=False)]

,alias,id,device,provider,size_mb,tools,license,task
44,qwen2.5-0.5b,qwen2.5-0.5b-instruct-cuda-gpu:4,GPU,CUDAExecutionProvider,528,True,apache-2.0,chat-completion
45,qwen2.5-0.5b,qwen2.5-0.5b-instruct-openvino-gpu:2,GPU,OpenVINOExecutionProvider,366,True,apache-2.0,chat-completion
46,qwen2.5-0.5b,qwen2.5-0.5b-instruct-generic-gpu:4,GPU,WebGpuExecutionProvider,700,True,apache-2.0,chat-completion
47,qwen2.5-0.5b,qwen2.5-0.5b-instruct-generic-cpu:4,CPU,CPUExecutionProvider,822,True,apache-2.0,chat-completion


### Let's use the qwen2.5-0.5b-instruct model
> https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct

In [12]:
# Download first (this may take a few minutes depending on model size)
manager.download_model("qwen2.5-0.5b-instruct-generic-gpu:4")

FoundryModelInfo(alias=qwen2.5-0.5b, id=qwen2.5-0.5b-instruct-generic-gpu:4, execution_provider=WebGpuExecutionProvider, device_type=GPU, file_size=700 MB, license=apache-2.0)

In [13]:
os.listdir(os.path.join(manager.get_cache_location(), "Microsoft"))

['Phi-4-mini-instruct-cuda-gpu-5', 'qwen2.5-0.5b-instruct-generic-gpu-4']

In [14]:
# Load a model into memory
model_info = manager.load_model("qwen2.5-0.5b-instruct-generic-gpu:4")
print(f"Loaded: {model_info.alias} => {model_info.id}")

Loaded: qwen2.5-0.5b => qwen2.5-0.5b-instruct-generic-gpu:4


In [15]:
info = manager.get_model_info("qwen2.5-0.5b-instruct-generic-gpu:4")
print(f"Running on: {info.execution_provider}")
print(f"Device:     {info.device_type}")

Running on: WebGpuExecutionProvider
Device:     GPU


In [16]:
# Initialize the OpenAI client
client = OpenAI(
    base_url=manager.endpoint,
    api_key=manager.api_key  # Not required for local usage
)

## Text Summarization Pipeline

Summarize articles, reports, or any text — all locally with zero data leaving your device.

In [17]:
def summarize(text: str,
              style: str = "concise",
              max_words: int = 2000) -> dict:
    """
    Summarize text with configurable style.

    Args:
        text: Input text to summarize.
        style: One of 'concise', 'bullet_points', 'eli5'.
        max_words: Approximate max words for summary.

    Returns:
        Dict with summary and metadata.
    """
    style_instructions = {
        "concise": f"Provide a concise summary in under {max_words} words.",
        "bullet_points":
        f"Summarize as 3-5 bullet points, each under 20 words.",
        "simple": f"Explain like I'm 5 years old, in under {max_words} words."
    }

    start = time.time()

    response = client.chat.completions.create(
        model=model_info.id,
        messages=[{
            "role": "system",
            "content": f"You are a summarization assistant. "
            f"{style_instructions.get(style, style_instructions['concise'])}"
        }, {
            "role": "user",
            "content": f"Summarize the following text:\n\n{text}"
        }],
        temperature=0.5,
        max_tokens=200)

    elapsed = time.time() - start

    return {
        "summary": response.choices[0].message.content,
        "style": style,
        "latency_sec": round(elapsed, 2)
    }

In [18]:
SAMPLE_TEXT = """
Mistral AI SAS (French: [mistʁal]) is a French artificial intelligence (AI) company, headquartered in Paris. Founded in 2023, it has open-weight large language models (LLMs), with both open-source and proprietary AI models.[2][3][4] As of 2025 the company has a valuation of more than US$14 billion.[5]

Namesake
The company is named after the mistral, a powerful, cold wind in southern France,[6] a term which originates from the Occitan language.

History
Mistral AI was established in April 2023 by three French AI researchers, Arthur Mensch, Guillaume Lample and Timothée Lacroix.[7]

Mensch, an expert in advanced AI systems, is a former employee of Google DeepMind; Lample and Lacroix, meanwhile, are large-scale AI models specialists who had worked for Meta Platforms.[8]

The trio originally met during their studies at École Polytechnique.[6]


Example of an image generated with Le Chat. The prompt is: Generate an image you feel represents yourself, Mistral AI.

Screenshot of Le Chat, Mistral AI chatbot, describing Wikipedia
Company operation
Funding
In June 2023, the start-up carried out a first fundraising of €105 million ($117 million) with investors including the American fund Lightspeed Venture Partners, Eric Schmidt, Xavier Niel and JCDecaux. The valuation was then estimated by the Financial Times at €240 million ($267 million).

On 10 December 2023, Mistral AI announced that it had raised €385 million ($428 million) as part of its second fundraising. This round of financing involves the Californian fund Andreessen Horowitz, BNP Paribas and the software publisher Salesforce.[9][10]

By December 2023, it was valued at over $2 billion.[11]

On 16 April 2024, reporting revealed that Mistral was in talks to raise €500 million, a deal that would more than double its current valuation to at least €5 billion.[12]

In June 2024, Mistral AI secured a €600 million ($645 million) funding round, increasing its valuation to €5.8 billion ($6.2 billion).[13] Based on valuation, as of June 2024, the company was ranked fourth globally in the AI industry, and first outside the San Francisco Bay Area.[14]

In August 2025, the Financial Times reported that Mistral was in talks to raise $1 billion at a $10 billion valuation.[15] In September 2025, Bloomberg announced that Mistral AI has secured a €2 billion investment valuing it at €12 billion ($14 billion).[16] This comes after $1.5 billion investment from Dutch company ASML, which owns 11% of Mistral.[17]

Partnerships
On 26 February 2024, Microsoft announced that Mistral's language models would be made available on Microsoft's Azure cloud, while the multilingual conversational assistant Le Chat would be launched in the style of ChatGPT.[18][non-primary source needed] The partnership also included a financial investment of $16 million by Microsoft in Mistral AI.[19]

In April 2025, Mistral AI announced a €100 million partnership with the shipping company CMA CGM.[20][21]
"""

In [19]:
print(SAMPLE_TEXT)


Mistral AI SAS (French: [mistʁal]) is a French artificial intelligence (AI) company, headquartered in Paris. Founded in 2023, it has open-weight large language models (LLMs), with both open-source and proprietary AI models.[2][3][4] As of 2025 the company has a valuation of more than US$14 billion.[5]

Namesake
The company is named after the mistral, a powerful, cold wind in southern France,[6] a term which originates from the Occitan language.

History
Mistral AI was established in April 2023 by three French AI researchers, Arthur Mensch, Guillaume Lample and Timothée Lacroix.[7]

Mensch, an expert in advanced AI systems, is a former employee of Google DeepMind; Lample and Lacroix, meanwhile, are large-scale AI models specialists who had worked for Meta Platforms.[8]

The trio originally met during their studies at École Polytechnique.[6]


Example of an image generated with Le Chat. The prompt is: Generate an image you feel represents yourself, Mistral AI.

Screenshot of Le Chat, Mi

In [20]:
# Try different summarization styles
for style in ["concise", "bullet_points", "simple"]:
    result = summarize(SAMPLE_TEXT, style=style)
    print(f"\n{'-'*100}")
    print(f"📝 Style: {style}")
    print(f"⏱️  Latency: {result['latency_sec']}s")
    print(f"\n{result['summary']}")
    print()


----------------------------------------------------------------------------------------------------
📝 Style: concise
⏱️  Latency: 14.83s

**Summary:**

**Mistral AI SAS** is a French artificial intelligence (AI) company founded in 2023. It specializes in developing **large language models**, such as **Open Source Models** and **Proprietary Models**, with both open-source and proprietary AI models. The company holds a market value exceeding **US$14 billion** as of 2025. 

**History:** 
- **Arthur Mensch** and **Guillaume Lample** were former employees of Google DeepMind.
- **Timothée Lacroix** and **Guillaume Lample** were large-scale AI experts specializing in Meta Platforms.

**Founders and Team:**
- **Arthur Mensch** - Former employee of Google DeepMind;
- **Guillaume Lample** and **Timothée Lacroix** - Large-scale AI experts;

**Acquisition:**
- **Arthur Mensch** and **Guillaume Lample** joined Mistral AI


--------------------------------------------------------------------------

## Batch Processing

Process multiple items efficiently with progress tracking and error handling.

In [21]:
def batch_process(items: list,
                  system_prompt: str,
                  max_tokens: int = 2000) -> list:
    """Process a batch of items with the local model."""
    results = []
    total = len(items)

    start_total = time.time()

    for i, item in enumerate(items):
        start = time.time()

        try:
            response = client.chat.completions.create(model=model_info.id,
                                                      messages=[{
                                                          "role":
                                                          "system",
                                                          "content":
                                                          system_prompt
                                                      }, {
                                                          "role": "user",
                                                          "content": item
                                                      }],
                                                      temperature=0.2,
                                                      max_tokens=max_tokens)

            results.append({
                "input": item,
                "output": response.choices[0].message.content,
                "latency": round(time.time() - start, 2),
                "status": "success"
            })
        except Exception as e:
            results.append({
                "input": item,
                "output": None,
                "error": str(e),
                "latency": round(time.time() - start, 2),
                "status": "error"
            })

        print(f"  [{i+1}/{total}] {item[:50]:50s} → {results[-1]['latency']}s "
              f"({'✅' if results[-1]['status'] == 'success' else '❌'})")

    total_time = time.time() - start_total
    print(f"\n📊 Batch complete: {total} items in {total_time:.1f}s "
          f"({total_time/total:.1f}s avg)")

    return results

In [22]:
# Example: Classify product reviews
reviews = [
    "This laptop is amazing! Great battery life and super fast.",
    "Service client terrible. Le produit ne marche plus au bout de 2 jours.",
    "Qualität ist okay für den Preis. Nichts Besonderes, aber funktioniert.",
    "¡Me encanta! La mejor compra que he hecho este año.",
    "Deludente. Il prodotto non corrisponde affatto alla descrizione.",
    "Boa relação qualidade-preço. Envio rápido e boa construção.",
]

In [23]:
for review in reviews:
    print(review)
    print()

This laptop is amazing! Great battery life and super fast.

Service client terrible. Le produit ne marche plus au bout de 2 jours.

Qualität ist okay für den Preis. Nichts Besonderes, aber funktioniert.

¡Me encanta! La mejor compra que he hecho este año.

Deludente. Il prodotto non corrisponde affatto alla descrizione.

Boa relação qualidade-preço. Envio rápido e boa construção.



In [24]:
print("Classifying product reviews...\n")

results = batch_process(
    reviews,
    system_prompt="Classify the following review as 'POS', 'NEG' or 'NEU.' "
    "Respond with just the label and a confidence score (0-1). "
    "Always answer in English "
    "Format: LABEL (confidence: X.X)")

print("\nResults:")
for r in results:
    if r["status"] == "success":
        print(f"\"{r['input'][:50]}\"  => {r['output']:30s}")

Classifying product reviews...

  [1/6] This laptop is amazing! Great battery life and sup → 0.45s (✅)
  [2/6] Service client terrible. Le produit ne marche plus → 0.44s (✅)
  [3/6] Qualität ist okay für den Preis. Nichts Besonderes → 0.46s (✅)
  [4/6] ¡Me encanta! La mejor compra que he hecho este año → 0.43s (✅)
  [5/6] Deludente. Il prodotto non corrisponde affatto all → 0.46s (✅)
  [6/6] Boa relação qualidade-preço. Envio rápido e boa co → 0.43s (✅)

📊 Batch complete: 6 items in 2.7s (0.4s avg)

Results:
"This laptop is amazing! Great battery life and sup"  => POS 0.98                      
"Service client terrible. Le produit ne marche plus"  => NEG 0.75                      
"Qualität ist okay für den Preis. Nichts Besonderes"  => NEU (0.98)                    
"¡Me encanta! La mejor compra que he hecho este año"  => POS 1.00                      
"Deludente. Il prodotto non corrisponde affatto all"  => NEG 0.75                      
"Boa relação qualidade-preço. Envio rápido e b

## Structured Data Extraction

Extract structured JSON from unstructured text — useful for ETL pipelines, document processing, and data entry automation.

In [25]:
def extract_structured_data(text: str, schema_description: str) -> dict:
    """Extract structured data from text using the local model."""
    response = client.chat.completions.create(
        model=model_info.id,
        messages=[{
            "role":
            "system",
            "content":
            f"You are a data extraction assistant. "
            f"Extract information and return ONLY valid JSON. "
            f"No markdown, no explanation, just the JSON object.\n\n"
            f"Expected schema: {schema_description}"
        }, {
            "role": "user",
            "content": f"Extract data from this text:\n\n{text}"
        }],
        temperature=0.7,
        max_tokens=2000)

    raw = response.choices[0].message.content.strip()

    # Try to parse as JSON
    try:
        # Remove potential markdown code fences
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
        return {"data": json.loads(raw), "raw": raw, "success": True}
    except json.JSONDecodeError:
        return {"data": None, "raw": raw, "success": False}

In [26]:
# Test with different extraction tasks
print("Test 1: Extract person info")

start = time.time()

result1 = extract_structured_data(
    "John Smith is a 35-year-old software engineer from Seattle. "
    "He works at TechCorp and has 10 years of experience in Python and Java.",
    schema_description=
    '{"name": str, "age": int, "city": str, "company": str, "skills": [str]}')

print(
    json.dumps(result1["data"], indent=2
               ) if result1["success"] else f"Parse failed: {result1['raw']}")

print(f"\n✅ Done in {(time.time() - start):.3f} seconds")

Test 1: Extract person info
{
  "name": "John Smith",
  "age": 35,
  "city": "Seattle",
  "company": "TechCorp",
  "skills": [
    "Python"
  ]
}

✅ Done in 0.845 seconds


In [27]:
print("Test 2: Extract product info")

start = time.time()

result2 = extract_structured_data(
    "The new Galaxy S25 Ultra features a 6.9-inch AMOLED display, "
    "Snapdragon 8 Elite chip, 12GB RAM, 256GB storage, and a 5000mAh battery. "
    "It costs $1,299 and is available in black and silver.",
    schema_description=
    '{"product": str, "display": str, "processor": str, "ram": str, '
    '"storage": str, "battery": str, "price": str, "colors": [str]}')

print(
    json.dumps(result2["data"], indent=2
               ) if result2["success"] else f"Parse failed: {result2['raw']}")

print(f"\n✅ Done in {(time.time() - start):.3f} seconds")

Test 2: Extract product info
{
  "product": "Galaxy S25 Ultra",
  "display": "AMOLED",
  "processor": "Snapdragon 8 Elite",
  "ram": "12GB",
  "storage": "256GB",
  "battery": "5000mAh",
  "price": "$1,299",
  "colors": [
    "black",
    "silver"
  ]
}

✅ Done in 1.617 seconds


In [28]:
print("Test 3: Extract event info")

start = time.time()

result3 = extract_structured_data(
    "Join us for the Annual AI Conference on March 15-17, 2026 "
    "at the Convention Center in Seattle, WA. Keynote speakers include "
    "Dr. Sarah Chen and Prof. James Miller. Tickets are $299 for early bird.",
    schema_description='{"event": str, "dates": str, "location": str, '
    '"speakers": [str], "price": str}')

print(
    json.dumps(result3["data"], indent=2
               ) if result3["success"] else f"Parse failed: {result3['raw']}")

print(f"\n✅ Done in {(time.time() - start):.3f} seconds")

Test 3: Extract event info
{
  "event": "Annual AI Conference",
  "dates": "March 15-17, 2026",
  "location": "Convention Center in Seattle, WA",
  "speakers": [
    "Dr. Sarah Chen",
    "Prof. James Miller"
  ],
  "price": "$299"
}

✅ Done in 1.375 seconds


## Simple RAG Pipeline (Retrieval-Augmented Generation)

Build a minimal RAG system using local files as a knowledge base. The model answers questions based on provided context — all running locally.

In [29]:
class SimpleLocalRAG:
    """
    A minimal RAG pipeline using Foundry Local.

    Strategy: keyword-based retrieval + LLM generation.
    For production, replace the retriever with a vector store.
    """

    def __init__(self, client, model_id: str):
        self.client = client
        self.model_id = model_info.id
        self.documents = []  # List of {"title": ..., "content": ...}

    def add_document(self, title: str, content: str):
        """Add a document to the knowledge base."""
        # Split into chunks for better retrieval
        chunk_size = 300  # words
        words = content.split()
        chunks = []

        for i in range(0, len(words), chunk_size):
            chunk_text = " ".join(words[i:i + chunk_size])
            chunks.append({
                "title": title,
                "content": chunk_text,
                "chunk_id": len(chunks)
            })

        self.documents.extend(chunks)
        print(f"  📄 Added '{title}': {len(chunks)} chunk(s)")

    def _retrieve(self, query: str, top_k: int = 3) -> list:
        """Simple keyword-based retrieval (replace with vector search in production)."""
        query_words = set(query.lower().split())

        scored = []
        for doc in self.documents:
            doc_words = set(doc["content"].lower().split())
            overlap = len(query_words & doc_words)
            scored.append((overlap, doc))

        scored.sort(key=lambda x: x[0], reverse=True)
        return [doc for score, doc in scored[:top_k] if score > 0]

    def query(self, question: str, verbose: bool = True) -> str:
        """Answer a question using the knowledge base."""
        # Retrieve relevant chunks
        relevant = self._retrieve(question)

        if verbose:
            print(f"🔍 Retrieved {len(relevant)} relevant chunk(s)")
            for r in relevant:
                print(f"   • [{r['title']}] {r['content'][:80]}...")

        if not relevant:
            context = "No relevant documents found in the knowledge base."
        else:
            context = "\n\n".join(f"[Source: {r['title']}]\n{r['content']}"
                                  for r in relevant)

        # Generate answer
        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=[{
                "role":
                "system",
                "content":
                "You are a knowledgeable assistant. Answer questions "
                "based ONLY on the provided context. If the context "
                "doesn't contain the answer, say so. Be concise."
            }, {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }],
            temperature=0.7,
            max_tokens=2000)

        return response.choices[0].message.content

In [30]:
# Build a knowledge base
rag = SimpleLocalRAG(client, model_info.id)

print("📚 Building knowledge base...\n")

rag.add_document(
    "Foundry Local Overview",
    """Foundry Local is an on-device AI inference solution from Microsoft that runs AI models 
    locally through a CLI, SDK, or REST API. It supports Windows 10/11, macOS, and various 
    hardware accelerators including NVIDIA CUDA GPUs, AMD GPUs, Intel NPUs, and Qualcomm NPUs. 
    The minimum requirement is 8 GB RAM and 3 GB disk space. It uses ONNX Runtime for 
    efficient model execution and provides an OpenAI-compatible API endpoint."""
)

rag.add_document(
    "Foundry Local Models",
    """Foundry Local supports multiple model families. Available models include Qwen 2.5 in 
    various sizes (0.5B, 1.5B, 3B, 7B parameters), Phi-4 mini, and Phi-3.5 mini. Models are 
    stored in ONNX format and automatically optimized for the user's hardware. The system 
    selects the best execution provider: CUDA for NVIDIA GPUs, QNN for Qualcomm NPUs, 
    OpenVINO for Intel devices, or CPU as fallback. Models that support tool calling include 
    Qwen 2.5-7B and Phi-4 mini.""")

rag.add_document(
    "Foundry Local Architecture",
    """Foundry Local runs a local REST server that exposes OpenAI-compatible endpoints. 
    The service dynamically assigns a port and can be managed through the CLI or SDK. 
    The FoundryLocalManager Python class handles service lifecycle, model downloading, 
    loading, and unloading. Models are cached locally after first download and can run 
    completely offline. The system supports streaming responses and tool calling."""
)

📚 Building knowledge base...

  📄 Added 'Foundry Local Overview': 1 chunk(s)
  📄 Added 'Foundry Local Models': 1 chunk(s)
  📄 Added 'Foundry Local Architecture': 1 chunk(s)


In [31]:
# Ask questions
questions = [
    "What hardware does Foundry Local support?",
    "Which models support tool calling?",
    "How does Foundry Local handle model selection?",
    "Can Foundry Local work offline?",
]

for q in questions:
    print(f"\n👤 {q}")
    start = time.time()
    answer = rag.query(q, verbose=False)
    print(f"🤖 {answer}")
    print(f"\n✅ Done in {(time.time() - start):.3f} seconds")
    print("-" * 100)


👤 What hardware does Foundry Local support?
🤖 Foundry Local supports multiple hardware types such as:

1. **NVIDIA CUDA GPUs**: This allows it to take advantage of GPU computing power.
2. **AMD GPUs**: Offers similar performance but may not have the same level of direct access to GPU memory compared to NVIDIA.
3. **Intel NPU** (e.g., Intel NVDIA): Provides better performance than GPU by directly accessing the GPU memory.
4. **Qualcomm NPU**: Similar to Intel NPU but offers higher efficiency due to direct access to the GPU memory.
5. **Custom Hardware Acceleration**: Can be customized with custom hardware components like CPUs.

So, while all these options are supported, the main emphasis here seems to be on GPU acceleration which aligns with its primary feature of running on-device AI models using NVIDIA GPUs.

✅ Done in 3.431 seconds
----------------------------------------------------------------------------------------------------

👤 Which models support tool calling?
🤖 Qwen 2.5

✅ 

# What we have learned?

## 🗂️ Model Catalog and Management

The local catalog exposes **over 80 model variants**, spanning CPU, CUDA GPU, OpenVINO, and WebGPU execution providers across multiple families including Phi, Qwen 2.5, Whisper, and GPT-OSS.

Model metadata can be easily converted into a **pandas DataFrame**, making filtering and inspection straightforward. The notebook demonstrated filtering by alias (e.g., Qwen 2.5), downloading a model once for local caching, and loading or unloading models into memory on demand.

> **Example model used:** `qwen2.5-0.5b-instruct` (GPU / WebGPU)

---

## 🔌 OpenAI-Compatible Local API

The standard **OpenAI Python client** connects directly to the local Foundry endpoint — no API key required. Because the API shape mirrors cloud OpenAI exactly, migrating existing code from cloud to local inference requires nothing more than changing the base URL.

---

## 📝 Text Summarization

A reusable local summarization pipeline was built to support multiple output styles: concise, bullet points, and simplified (ELI5). Per-request latency was measured across runs, and the pipeline was validated against long, real-world text passages.

All inference ran **entirely locally with zero data egress**.

---

## ⚡ Batch Processing

Batch inference was implemented with built-in progress tracking, per-item latency measurement, and error handling. The example use case — **multilingual sentiment classification of product reviews** — produced structured output including a label (positive, negative, or neutral) and a confidence score.

Results showed **predictable and fast local performance at scale**.

---

## 🧩 Structured Data Extraction (JSON)

Strict JSON was extracted from unstructured text by enforcing schemas through system prompts, with automatic parsing and validation. The notebook demonstrated extraction across several domains: person profiles, product specifications, and event details.

This pattern is well suited for ETL pipelines, document processing, and automation workflows where structured output from free-form text is essential.

---

## 🔍 Simple Local RAG (Retrieval-Augmented Generation)

A minimal RAG pipeline was built to run **fully locally**, using keyword-based retrieval against a set of local documents as the knowledge base — no vector database required. Answers were grounded strictly in the retrieved context, with a clear separation between the retrieval and generation stages.

The retrieval component can be easily swapped for an embedding-based approach in production scenarios.

> Go to the next notebook